# EAS Temperament PLS with Diagnostic Measures
## Canonical PLS Analysis of FEMA-residualized Phenotypes

**Purpose:** Perform canonical partial least squares (PLS) to identify multivariate patterns linking EAS temperament deviations from longitudinal developmental trajectories with diagnostic categories.

**Methodology:**
- Load FEMA mixed-effects model results from baseline analysis
- Extract fixed-effect predictions (developmental trajectories) and residuals (individual deviations)
- Compute trajectory deviation scores (average residuals across time points)
- Perform PLS on deviations vs. diagnostic data
- Use family-aware cluster bootstrap resampling to obtain robust confidence intervals
- Conduct permutation testing with family structures preserved to assess significance

**Outputs:** 
- PLS loading vectors, bootstrap distributions, and confidence intervals
- Significance indicators for each canonical dimension
- Permutation p-values assessing canonical correlations


In [ ]:
# ========================================================================
# Import required libraries
# ========================================================================
import pandas as pd  # Data manipulation and I/O
import numpy as np  # Numerical computing
from sklearn.decomposition import PCA  # Dimensionality reduction
import h5py  # HDF5 file I/O for MATLAB .mat files
from sklearn.linear_model import LinearRegression  # Not used but available
from sklearn.preprocessing import StandardScaler  # Z-score standardization
from sklearn.cross_decomposition import PLSCanonical  # PLS canonical correlation
from sklearn.utils import resample  # Bootstrap resampling utilities
from numpy.random import default_rng  # Random number generation with seeds


In [ ]:
# Linear regression model
model = LinearRegression()
scaler = StandardScaler()

In [ ]:
def fix_dims(var, name):
    """
    Correct array dimension ordering from MATLAB HDF5 export to Python conventions.
    
    MATLAB and Python have different array memory layouts (column-major vs row-major),
    requiring axis transposition for proper interpretation. This function handles
    common dimension patterns from FEMA output.
    
    Parameters:
    -----------
    var : np.ndarray
        Array to be re-oriented
    name : str
        Variable name for diagnostic messages
        
    Returns:
    --------
    np.ndarray
        Re-oriented array with corrected axis order
        
    Dimension mapping:
    - 1D: No change (already correct)
    - 2D: Transpose to (p,v) → (v,p)
    - 4D: Expected FEMA sig2mat format (v,r,k,k) → (k,k,r,v)
    - 3D: Warning message; manual verification recommended
    """
    if var.ndim == 1:
        return var  # Likely already correct
    elif var.ndim == 2:
        return var.T  # Flip from (v, p) → (p, v)
    elif var.ndim == 4:
        return np.transpose(var, (2, 3, 1, 0))  # Assume FEMA sig2mat: (v, r, k, k) → (k, k, r, v)
    elif var.ndim == 3:
        # Handle if you expect 3D arrays like [p, v, rep] → [rep, p, v] or similar
        print(f"Warning: 3D array {name} — please verify axis order manually.")
        return var
    else:
        print(f"Unhandled array shape {var.shape} for variable {name}")
        return var


Subject data

In [ ]:
# ========================================================================
# LOAD PHENOTYPIC DATA
# ========================================================================
# Load main data CSV containing all demographic, genetic, and outcome variables
df = pd.read_csv("....csv")

# Define genetic Principal Component column names (PC1-PC20)
# These PCs control for population stratification and ancestry effects
pc_cols = [f"PC{i}" for i in range(1, 21)]

# Identify genotyping batch columns (technical covariates for batch effects)
batch_cols = [col for col in df.columns if 'genotyping_batch' in col]

# Define longitudinal time point labels (18, 36, 60 months)
time_points = ['18_months', '36_months', '60_months']

# Define EAS temperament outcome variables:
# - emot: Emotional reactivity
# - act:  Activity level  
# - shy:  Shyness
# - soc:  Sociability
phenNames = ['emot', 'act', 'shy', 'soc']


## Load FEMA Baseline Model Results
Load variance components and fixed-effect estimates from the longitudinal mixed-effects model fit to EAS data. These provide the developmental trajectories and residuals used downstream.


In [ ]:
def load_fema_hdf5(file_path):
    """
    Load FEMA results from HDF5/MAT file with proper handling of MATLAB-specific data types.
    
    MATLAB exports object references and character arrays that require special decoding.
    This function recursively loads HDF5 groups/datasets while handling these types.
    
    Parameters:
    -----------
    file_path : str
        Path to HDF5 file exported from MATLAB (typically FEMA.mat)
        
    Returns:
    --------
    dict
        Dictionary containing all FEMA output variables:
        - IID: Individual identifiers
        - EID: Event time points (1,2,3 for 18/36/60 months)
        - FID: Family identifiers
        - beta_hat: Fixed effect estimates (covariate coefficients)
        - X: Covariate design matrix
        - sig2mat: Variance-covariance matrices (random effects)
        - And other FEMA diagnostic quantities
    """
    data = {}

    def load_dataset_or_group(obj):
        """Recursively load datasets, decode MATLAB references and char arrays."""
        if isinstance(obj, h5py.Dataset):
            if obj.dtype == 'O':  # MATLAB object reference - requires dereferencing
                deref = []
                for ref in obj[:].squeeze():
                    ref_obj = f[ref]
                    if ref_obj.shape == ():  # scalar string
                        s = ref_obj[()].decode("utf-8")
                    else:  # char array - concatenate characters
                        s = ''.join(chr(c[0]) for c in ref_obj[:])
                    deref.append(s)
                return deref
            else:
                return obj[()]  # Regular numeric dataset
        elif isinstance(obj, h5py.Group):
            # Recursively load all keys in this group
            return {k: load_dataset_or_group(obj[k]) for k in obj.keys()}
        else:
            return obj

    with h5py.File(file_path, 'r') as f:
        for key in f.keys():
            # Skip MATLAB internal references and subsystems
            if key.startswith('#refs#') or key.startswith('#subsystem#'):
                continue
            try:
                data[key] = load_dataset_or_group(f[key])
                print(f"Loaded: {key} | Type: {type(data[key])}")
            except Exception as e:
                print(f"Failed to load {key}: {e}")

    return data


In [ ]:
# Load FEMA results from HDF5 file (converted from MATLAB .mat output)
data = load_fema_hdf5("/FEMA.mat")


In [ ]:
# Reorganize FEMA data: convert from MATLAB to Python array conventions
# This ensures correct axis alignment for downstream analysis
fema = {}
for key, val in data.items():
    if isinstance(val, np.ndarray):
        fema[key] = fix_dims(val, key)  # Transpose axes as needed
    else:
        fema[key] = val  # Keep non-array types unchanged


In [ ]:
# Extract fixed effect estimates (regression coefficients for covariates)
# beta shape: (n_covariates, n_phenotypes) - used for predictions
beta = fema['beta_hat']


## Load Diagnostic Data
Load diagnostic categorizations or dimensional scores to use as external validation and association targets for PLS.


In [ ]:
# Load diagnostic data
df_diag = pd.read_csv('....csv')

# Reindex df_diag to match the index of phenotype dataframe
# Missing rows will be filled with NaN, replaced with 0 (zero = missing category)
df_diag_subset = df_diag.reindex(np.unique(df.index)).fillna(0)

# Drop columns that contain only zeros (missing for all subjects)
df_diag_subset = df_diag_subset.loc[:, (df_diag_subset != 0).any(axis=0)]


In [ ]:
# Subset phenotype data by visit/event time point
# Each time point represents a separate longitudinal assessment
df_18 = df[df.eid == '18_months']  # Baseline visit (18 months)
df_36 = df[df.eid == '36_months']  # Follow-up 1 (36 months)
df_60 = df[df.eid == '60_months']  # Follow-up 2 (60 months)


In [ ]:
def zscore_dataframe(df, phenNames):
    """
    Z-score standardize phenotype columns within a time point or subset.
    
    Standardization: X_standardized = (X - mean(X)) / std(X)
    Ensures all variables on same scale (mean=0, std=1) for statistical modeling.
    
    Parameters:
    -----------
    df : pd.DataFrame
        Input dataframe with phenotype columns
    phenNames : list
        Column names to standardize (EAS outcomes)
        
    Returns:
    --------
    pd.DataFrame
        Standardized phenotypes with preserved index and column names
    """
    scaler = StandardScaler()

    # Extract phenotype values
    X = df.loc[:, phenNames].values

    # Apply z-score transformation
    X_ss = scaler.fit_transform(X)

    # Return as DataFrame preserving index and column structure
    df_res = pd.DataFrame(data=X_ss, index=df.index, columns=phenNames)
    return df_res

# Apply standardization to each time point separately
# (within-timepoint standardization ensures comparable scales across visits)
df_ss_18 = zscore_dataframe(df_18, phenNames)
df_ss_36 = zscore_dataframe(df_36, phenNames)
df_ss_60 = zscore_dataframe(df_60, phenNames)


In [ ]:
df_ss_18 = zscore_dataframe(df_18, phenNames)
df_ss_36 = zscore_dataframe(df_36, phenNames)
df_ss_60 = zscore_dataframe(df_60, phenNames)

## Predict FEMA Fixed Effects (Developmental Trajectories)
Generate predicted phenotypes based on fixed effects only (covariates: age smooth trend, sex, ancestry PCs, batch).
These represent the expected developmental trajectory for each subject at each time point.
Residuals capture individual-specific deviations from these trajectories.


In [ ]:
# Construct data frame with FEMA fixed effect design matrix
# IID and EID: subject and event identifiers from FEMA
df_fema_raw = pd.DataFrame({'iid': fema['IID'], 'eid': fema['EID'].squeeze()})

# Extract covariate columns from FEMA design matrix X 
# (intercept + spline basis for age + sex + genetic PCs + batch)
df_tmp = pd.DataFrame(fema['X'])
df_tmp.columns = ['X' + str(col) for col in df_tmp.columns]

# Combine identifiers with covariate matrix
df_fema_raw = pd.concat([df_fema_raw, df_tmp], axis=1, ignore_index=False)


In [ ]:
def predict_FEMA_fixed(df, beta, timepoint, phenNames):
    """
    Predict phenotypes from FEMA fixed effects (developmental trajectory).
    
    Prediction = X @ beta, where:
    - X: covariate design matrix (includes age smooth trend, sex, ancestry, batch)
    - beta: fitted fixed effect coefficients
    
    Only includes fixed effects; random effects (individual/family deviations) are excluded.
    
    Parameters:
    -----------
    df : pd.DataFrame
        Data frame containing IID, EID, and X columns
    beta : np.ndarray
        Fixed effect estimates (n_covariates x n_phenotypes)
    timepoint : int
        Event ID (1=18mo, 2=36mo, 3=60mo)
    phenNames : list
        Phenotype outcome names
        
    Returns:
    --------
    pd.DataFrame
        Predicted phenotype values indexed by individual ID
    """
    # Select data for this time point
    df_red = df[df.eid == timepoint]
    
    # Extract covariate matrix X for selected records
    X = df_red.loc[:, df_red.columns.str.startswith('X')].to_numpy()
    
    # Compute prediction: X @ beta (matrix multiplication)
    y_pred = X @ beta
    
    # Return as DataFrame with IID index and phenotype names
    df_red = pd.DataFrame(data=y_pred, index=df_red.iid, columns=phenNames)
    df_red = df_red.drop(columns='iid', errors='ignore')
    return df_red

# Predict fixed effect trajectories for each visit
df_fema_18 = predict_FEMA_fixed(df_fema_raw, beta, 1, phenNames)
df_fema_36 = predict_FEMA_fixed(df_fema_raw, beta, 2, phenNames)
df_fema_60 = predict_FEMA_fixed(df_fema_raw, beta, 3, phenNames)


## Individual Deviations from Trajectories

Calculate residuals by subtracting predicted trajectories from observed phenotypes.


In [ ]:
def predict_FEMA_fixed_diff(df, df_fema, phenNames):
    """
    Calculate residuals from FEMA fixed effects.
    
    Residuals = Observed - Predicted(fixed effects)
    
    These deviations capture departures from the population-average developmental trajectory.
    
    Parameters:
    -----------
    df : pd.DataFrame
        Frame with standardized observed phenotypes
    df_fema : pd.DataFrame
        Frame with FEMA fixed effect predictions
    phenNames : list
        Phenotype column names
        
    Returns:
    --------
    pd.DataFrame
        Residual values (observed minus predicted)
    """
    df_diff = df.loc[:, phenNames] - df_fema.loc[:, phenNames]
    return df_diff

# Compute residuals (random effects) for each time point
df_fema_diff_18 = predict_FEMA_fixed_diff(df_ss_18, df_fema_18, phenNames)
df_fema_diff_36 = predict_FEMA_fixed_diff(df_ss_36, df_fema_36, phenNames)
df_fema_diff_60 = predict_FEMA_fixed_diff(df_ss_60, df_fema_60, phenNames)


## Trajectory Deviation Score: Individual-level Phenotypic Summary
Compute a subject-level deviation score by averaging residuals across all longitudinal time points.
This single value per subject per trait quantifies how much the individual deviates from their expected trajectory.


In [ ]:
# Find individuals present across all three time points (complete cases)
# Only use subjects with observations at all visits for consistent comparison
common_indices = df_fema_18.index.intersection(df_fema_36.index).intersection(df_fema_60.index)

# Align all residual dataframes to these common subjects in consistent order
df_fema_diff_18_aligned = df_fema_diff_18.loc[common_indices].sort_index()
df_fema_diff_36_aligned = df_fema_diff_36.loc[common_indices].sort_index()
df_fema_diff_60_aligned = df_fema_diff_60.loc[common_indices].sort_index()


In [ ]:
# Stack residuals into 3D array: (n_subjects, n_timepoints, n_traits)
# This prepares data for longitudinal analysis and averaging
residuals = np.stack([df_fema_diff_18_aligned.values,
                      df_fema_diff_36_aligned.values,
                      df_fema_diff_60_aligned.values], axis=1)  # shape: (subjects, time, traits)

# Average residuals across all timepoints to obtain a single trajectory deviation score per subject
# This summary score indicates overall tendency to deviate from expected developmental path
trajectory_deviation_score = residuals.mean(axis=1)  # shape: (subjects, traits)

# Convert to DataFrame for easier handling and interpretation with trait names
df_deviation = pd.DataFrame(trajectory_deviation_score, columns=phenNames, index=df_fema_diff_18_aligned.index)


## Canonical PLS Analysis: Link Trajectory Deviations to Diagnoses
Perform PLS to identify multivariate associations between EAS trajectory deviations and
diagnoses. PLS maximizes covariance between the two domains while maintaining
interpretability through loadings (correlations with original variables).


In [ ]:
# Prepare analysis data: use trajectory deviation scores as X domain (EAS)
df = df_deviation.copy()
df_diag_subset = df_diag.copy()

# Reindex diagnostic data to match phenotype subjects; fill missing with 0
df_diag_subset = df_diag_subset.reindex(np.unique(df.index)).fillna(0)

# Extract diagnostic category identifiers (first 1-2 letters of column names)
df_cats = df_diag_subset.columns[1:].str[0:2]

# Find subjects with both EAS and diagnostic data available
common_index = df.index.intersection(df_diag_subset.index)

# Extract matrices for CCA
# x: EAS trajectory deviations (n_subjects x n_traits)
# y: Diagnostic data (n_subjects x n_diagnostics)
x = df.loc[common_index].values
y = df_diag_subset.loc[common_index, :].values

# Reduce diagnostic space to 100 dimensions via PCA (for computational stability)
# Prevents overfitting and improves numerical conditioning
pca = PCA(n_components=100)
y_pca = pca.fit_transform(y)


## Bootstrap PLS Loadings with Family-Aware Resampling
Perform cluster bootstrap to estimate sampling distributions of PLS loadings while respecting family structure.
Family members kept together in each bootstrap sample to preserve kinship correlations.
Loadings = correlations between original variables and their canonical variates.
Used to determine which variables contribute to each canonical dimension.


In [ ]:
# ========================================================================
# Run PLS Analysis (Main Model)
# ========================================================================
# Fit PLS with 3 components on full data to identify canonical dimensions
ndim = 3
pls = PLSCanonical(n_components=ndim, scale=False, max_iter=10000)
pls.fit(x, y_pca)

# Extract rotation matrices (weight vectors for computing canonical variates)
x_rotations = pls.x_rotations_  # EAS weights
y_rotations = pls.y_rotations_  # Diagnosis weights

# Inverse transform y_rotations back to original diagnostic variable space
# (Since PCA was applied to y, must invert to get loadings in diagnostic coordinates)
y_rotations = pca.inverse_transform(y_rotations.T).T

# Compute canonical variates (linear combinations of original variables)
# scores = original_data @ rotation_weights
x_score = np.dot(x, x_rotations)  # EAS canonical variates
y_score = np.dot(y, y_rotations)  # Diagnosis canonical variates

# ========================================================================
# Compute PLS Loadings (Correlations with Canonical Variates)
# ========================================================================
# Loadings = correlation(original variable, canonical variate)
# Indicators of variable contribution to each canonical dimension
x_loadings = np.zeros((np.shape(x)[1], ndim))
y_loadings = np.zeros((np.shape(y)[1], ndim))

for i in range(ndim):
    # Pearson correlation between variable values and canonical scores
    x_loadings[:, i] = np.array([np.corrcoef(x[:, j], x_score[:, i])[
                                0, 1] for j in range(np.shape(x)[1])])
    y_loadings[:, i] = np.array([np.corrcoef(y[:, j], y_score[:, i])[
                                0, 1] for j in range(np.shape(y)[1])])


In [ ]:
# Extract family structure from phenotype data
# Family information needed for cluster bootstrap (keep families together in resampling)
family_index = df_18.FID.loc[common_indices].sort_index().values
family_index = np.asarray(family_index)  # shape (n,) - family ID for each subject

# Create mapping: which rows belong to each unique family
families = np.unique(family_index)
fam_rows = {fam: np.flatnonzero(family_index == fam) for fam in families}


In [ ]:
def cluster_bootstrap_indices(family_index, rng):
    """
    Generate bootstrap sample indices respecting family clusters.
    
    Standard bootstrap resampling from individuals would break family structure
    and violate kinship correlations. This function resamples families (not individuals)
    with replacement, keeping all family members together.
    
    Parameters:
    -----------
    family_index : np.ndarray
        Family ID for each subject (shape: n_subjects)
    rng : np.random.Generator
        Seeded random number generator for reproducibility
        
    Returns:
    --------
    np.ndarray
        Bootstrap indices preserving family clustering
    """
    # Identify unique families
    families = np.unique(family_index)
    fam_rows = {fam: np.flatnonzero(family_index == fam) for fam in families}

    # Sample families with replacement (cluster bootstrap)
    # Each family equally likely to be selected
    sampled_fams = rng.choice(families, size=len(families), replace=True)

    # Concatenate rows for sampled families (siblings kept together)
    boot_idx = np.concatenate([fam_rows[f] for f in sampled_fams])
    return boot_idx


In [ ]:
# ========================================================================
# BOOTSTRAP PROCEDURE: Estimate Sampling Distribution of PLS Loadings
# ========================================================================
# Iterate 10000 times: resample families, refit PLS, compute loadings,
# align with base model to ensure consistency across iterations

nboot = 10000  # Number of bootstrap resamples
ndim = 2  # Number of canonical dimensions (test 2 for next section, originally 3)
pls_boot = PLSCanonical(n_components=ndim+1, scale=False, max_iter=10000)  # +1 to detect outlier dimensions
x_loadings_boot = []  # Storage for bootstrap loadings
y_loadings_boot = []

# Initialize
i = 0  # Bootstrap iteration counter
cct = 0  # Tracking convergence/issues
ccq = 0  # Tracking alignment failures
rng = default_rng(0)

# Bootstrap loop with adaptive stopping
while len(x_loadings_boot) < nboot:
    i = i + 1
    rng_i = default_rng(i)  # Seed with iteration number for reproducibility
    
    # Resample families with replacement (keeps family members together)
    idx = cluster_bootstrap_indices(family_index, rng_i)
    x_boot = x[idx, :]  # EAS data for bootstrap sample
    y_boot = y[idx, :]  # Diagnosis data for bootstrap sample

    # Apply PCA to bootstrap diagnostic data (must be done per sample)
    y_boot_pca = pca.fit_transform(y_boot)
    
    # Fit PLS to bootstrap sample
    pls_boot.fit(x_boot, y_boot_pca)

    # Extract rotations from bootstrap fit
    x_rotations_boot = pls_boot.x_rotations_  # EAS weights
    y_rotations_boot = pca.inverse_transform(pls_boot.y_rotations_.T).T  # Inverse transform to diagnostic space

    # Compute canonical variates for FULL sample using bootstrap rotation weights
    # (This is important: evaluate rotations on full sample to compare across iterations)
    x_score_boot = np.dot(x, x_rotations_boot)
    y_score_boot = np.dot(y, y_rotations_boot)

    # ====================================================================
    # Compute Loadings for This Bootstrap Iteration
    # ====================================================================
    # Loadings = correlations between original variables and their canonical variates
    tmp_x_loadings_boot = np.zeros((np.shape(x)[1], ndim+1))
    tmp_y_loadings_boot = np.zeros((np.shape(y)[1], ndim+1))
    
    for j in range(ndim+1):
        tmp_x_loadings_boot[:, j] = np.array([np.corrcoef(x[:, k],
                                                          x_score_boot[:, j])[0, 1] for k in range(np.shape(x)[1])])
        tmp_y_loadings_boot[:, j] = np.array([np.corrcoef(y[:, k],
                                                          y_score_boot[:, j])[0, 1] for k in range(np.shape(y)[1])])

    # ====================================================================
    # Align Bootstrap Loadings to Base Model Loadings
    # ====================================================================
    # Canonical dimensions are sign invariant; must match each dimension in the bootstrap
    # to its corresponding dimension in the base model to ensure consistency
    
    flag = []   # Sign flip needed (+1 or -1) to match base dimension
    
    for j in range(ndim):
        # Sign of correlation (flip if negative)
        flag.append(np.sign(np.corrcoef(x_loadings[:, j], tmp_x_loadings_boot[:, j])[0, 1]))
 
    # Flip signs if needed to match directionality of base model
    for j in range(ndim):
        tmp_x_loadings_boot[:, j] = flag[j] * tmp_x_loadings_boot[:, j]
        tmp_y_loadings_boot[:, j] = flag[j] * tmp_y_loadings_boot[:, j]

    # Store this bootstrap iteration's aligned loadings
    x_loadings_boot.append(tmp_x_loadings_boot)
    y_loadings_boot.append(tmp_y_loadings_boot)

# Convert list of arrays to stacked array: (nboot, n_variables, ndim)
x_loadings_boot = np.array(x_loadings_boot)
y_loadings_boot = np.array(y_loadings_boot)

print(f"Bootstrap iterations accepted: {len(x_loadings_boot)}/{i}")
print(f"Alignment failures: {ccq}")


In [ ]:
# ========================================================================
# Compute Bootstrap Confidence Intervals and Significance
# ========================================================================
# Extract 2.5th and 97.5th percentiles of bootstrap loadings for each variable/dimension
# A loading is "significant" if the 95% CI doesn't cross zero
# (i.e., lower and upper bounds have the same sign)

x_loadings_sig = []  # Binary: significant (1) or not (0) for each x variable/dimension
y_loadings_sig = []

x_loadings_low = np.zeros((np.shape(x)[1], ndim))   # 2.5th percentile
x_loadings_high = np.zeros((np.shape(x)[1], ndim))  # 97.5th percentile
y_loadings_low = np.zeros((np.shape(y)[1], ndim))
y_loadings_high = np.zeros((np.shape(y)[1], ndim))

for i in range(ndim):
    # Extract percentiles for EAS loadings
    low, high = np.percentile(x_loadings_boot[:, :, i], [2.5, 97.5], axis=0)
    
    # Significant if both bounds have same sign: (sign(high) * sign(low) > 0)
    # This means CI doesn't include zero
    x_loadings_sig.append(((np.sign(high) * np.sign(low)) > 0)*1)
    x_loadings_low[:, i] = low
    x_loadings_high[:, i] = high

    # Extract percentiles for diagnostic loadings
    low, high = np.percentile(y_loadings_boot[:, :, i], [2.5, 97.5], axis=0)
    y_loadings_sig.append(((np.sign(high) * np.sign(low)) > 0)*1)
    y_loadings_low[:, i] = low
    y_loadings_high[:, i] = high

# Convert significance arrays to matrices (n_variables x ndim)
x_loadings_sig = np.array(x_loadings_sig).T
y_loadings_sig = np.array(y_loadings_sig).T


## Determine Optimal Number of PLS Dimensions via Permutation Testing
Test the significance of each canonical dimension by permuting family structures and
comparing observed canonical covariances to null distribution.
Determines how many dimensions to retain for final model.


In [ ]:
# Extract family structure from the diagnostic data frame for permutation testing
# (Different source than earlier; ensure consistency)
family_index = df_18.FID.loc[common_indices].sort_index().values


In [ ]:
# Compute observed canonical correlations (covariances between canonical variates)
# These are the test statistics for permutation test
pval = []
actual_cov = []
n_permutations = 10000  # Number of permutations

for i in range(ndim):
    # Canonical correlation = covariance(x_scores, y_scores) for dimension i
    actual_cov.append(np.cov(x_score[:, i], y_score[:, i])[0, 1])
actual_cov = np.array(actual_cov)


In [ ]:
# Prepare family structure for cluster-aware permutation
family_index = np.asarray(family_index)  # shape (n,) - family ID for each subject

# Map unique families to their row indices
families, inv = np.unique(family_index, return_inverse=True)
fam_rows = [np.flatnonzero(inv == k) for k in range(len(families))]


In [ ]:
# ========================================================================
# PERMUTATION TEST: Null Distribution of Canonical Covariances
# ========================================================================
# For each permutation:
# 1. Resample family order (break association between X and Y domains)
# 2. Keep family members together (preserve within-family correlations)
# 3. Refit PLS and compute canonical covariance
# 4. Compare to observed covariance to get p-values

permuted_covs = np.empty(n_permutations, dtype=float)

for i in range(n_permutations):
    rng = np.random.default_rng(i)  # Seed with iteration for reproducibility

    # Permute family ORDER (not individual order)
    # This breaks the association between EAS (x) and diagnosis (y) domains
    fam_order = rng.permutation(len(fam_rows))  # Random family ordering
    perm_rows = np.concatenate([fam_rows[k] for k in fam_order])  # Indices for this permutation

    # Permute y-space data using family-level permutation
    # x remains in original order; y is reordered by families
    # This breaks x-y association while preserving family structures in y
    y_perm = y_pca[perm_rows, :]  # Reorder EAS data by permuted families

    # Fit PLS to permuted data (x original order, y permuted order)
    x_scores_perm, y_scores_perm = pls.fit_transform(x, y_perm)
    
    # Compute first canonical covariance for this permutation
    permuted_covs[i] = np.cov(x_scores_perm[:, 0], y_scores_perm[:, 0])[0, 1]

# ========================================================================
# Compute p-values: proportion of permutations with covariance >= observed
# ========================================================================
pval = []
for d in range(ndim):
    # Two-tailed test: count permutations with |cov| >= |observed cov|
    # Add 1 to numerator and denominator (standard in permutation testing)
    pval.append((np.sum(np.abs(permuted_covs) >= np.abs(actual_cov[d])) + 1) / (n_permutations + 1))

print(f"Canonical covariance p-values: {pval}")
print(f"Significant dimensions (p < 0.05): {[i for i, p in enumerate(pval) if p < 0.05]}")
